# Direct (chain-reducer) vs explicit decay

Demonstrates the two ways PriNCe handles unstable mothers (free n, H-3, charged mesons, ...):

- **Chain reducer (default):** unstable species are folded into their stable daughters at cross-section build time. `_reduce_channels` walks each unstable mother to its terminal stable products and rewrites the channel inclusively. The state vector never carries `n`, `π±`, etc. — their decay is treated as instantaneous on every photo-disintegration step.

- **Explicit decay:** unstable mothers live in the state vector and decay via the ETD2 solver's Λ operator. Diagonal block Λ_diag carries the lifetime; off-diagonal block Λ_off carries the daughter redistribution. Activated by `config.enable_explicit_decay=True` plus `enable_decay=True` on the solver.

Physics expectation: the two modes agree wherever the lab-frame decay length γ·c·τ is short on cosmological scales. They diverge at the high-energy end (e.g. γ_n ≳ 10⁹ → lab decay length ≳ Mpc) where instant decay over-predicts daughter production.

Setup mirrors the explicit-decay end-to-end test (`tests/test_explicit_decay_vs_chain_reducer.py`): `max_mass=4`, He-4 source with an Auger-fit-like spectrum, propagation z=1→0 on the scipy backend. Spectra and residuals are plotted across the full energy grid (no down-sampling).

In [ ]:
# Point PriNCe at the FLUKA-derived photo-nuclear database (v3 prod).
# Built by prince-fluka-utils — replaces the SOPHIA + PEANUT/TALYS
# split. Override fluka_db_path / fluka_db_fname for a different machine.
import prince_cr.config
prince_cr.config.fluka_db_path = '/home/anatoli/work/devel/UH-UHECR-Fluka-Prince/runs/2026-05-06_pfu-v3-prod/outputs/'
prince_cr.config.fluka_db_fname = 'prince_db_v3-sparse.h5'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from prince_cr import core, photonfields, cross_sections
from prince_cr.cr_sources import AugerFitSource
from prince_cr.solvers import UHECRPropagationSolverETD2

In [ ]:
# Standard PriNCe physics knobs. tau_dec_threshold=inf is required
# so unstable species pass the lifetime gate; the chain-reducer flag
# (config.enable_explicit_decay) is what actually selects the mode.
prince_cr.config.x_cut = 1e-4
prince_cr.config.x_cut_proton = 1e-2
prince_cr.config.tau_dec_threshold = np.inf

## Build a kernel for each mode

`config.enable_explicit_decay` is read by `CrossSectionBase._reduce_channels` at the cross-section's `_optimize_and_generate_index` step — which runs once per cross-section instance. So we need a fresh `cross_sections.FlukaPhotoNuclear()` and a fresh `PriNCeRun` for each mode. The flag is saved/restored around each build so the rest of the session is unaffected.

In [ ]:
pf = photonfields.CombinedPhotonField(
    [photonfields.CMBPhotonSpectrum, photonfields.CIBGilmore2D]
)

In [ ]:
def make_run(*, enable_explicit_decay, max_mass=4):
    saved = getattr(prince_cr.config, 'enable_explicit_decay', False)
    prince_cr.config.enable_explicit_decay = enable_explicit_decay
    try:
        cs = cross_sections.FlukaPhotoNuclear()
        return core.PriNCeRun(max_mass=max_mass, photon_field=pf, cross_sections=cs)
    finally:
        prince_cr.config.enable_explicit_decay = saved

In [ ]:
%time prince_run_chain = make_run(enable_explicit_decay=False)

In [ ]:
%time prince_run_expl = make_run(enable_explicit_decay=True)

## Solve both modes

Auger-fit He-4 source, `z=1→0`, scipy linear-algebra backend, `dz=1e-3` (1000 ETD2 steps). At `dz=1e-2` integration error is ~5 % at the high-E end of the strict band; `dz=1e-3` gets us under 0.5 % so the residual is dominated by the chain-reducer's instant-decay approximation rather than time-stepping.

In [ ]:
# Auger-fit He-4 only. Same source params for both modes; the only
# difference between the runs is the decay treatment.
AUGER_HE4 = {1000020040: (0.96, 10**9.68, 50.0)}


def make_solver(prince_run, *, enable_decay):
    prince_run.backend.linear_algebra_backend = 'scipy'
    s = UHECRPropagationSolverETD2(
        initial_z=1.0,
        final_z=0.0,
        prince_run=prince_run,
        enable_pairprod_losses=True,
        enable_adiabatic_losses=True,
        enable_injection_jacobian=True,
        enable_partial_diff_jacobian=True,
        enable_decay=enable_decay,
    )
    s.add_source_class(AugerFitSource(prince_run, norm=1e-50, params=AUGER_HE4))
    return s

In [ ]:
solver_chain = make_solver(prince_run_chain, enable_decay=False)
%time solver_chain.solve(dz=1e-3, verbose=False, progressbar=True)

In [ ]:
solver_expl = make_solver(prince_run_expl, enable_decay=True)
%time solver_expl.solve(dz=1e-3, verbose=False, progressbar=True)

# Sanity: explicit-decay mode should have populated Λ.
assert solver_expl._etd2_Lambda_diag is not None, 'Λ_diag was not built'
assert solver_expl._etd2_Lambda_off is not None, 'Λ_off was not built'

## Extract spectra at z=0

PriNCe's state vector is `dN/dE` on the log energy grid. In explicit-decay mode the surviving free `n` is a separate species (PDG 2112), so the apples-to-apples nucleon comparison is `chain[p]` vs `explicit[p + n]`.

In [ ]:
def get_spectrum(state, spec_man, pdgid):
    s = spec_man.pdgid2sref.get(pdgid)
    if s is None:
        return None
    return state[s.lidx():s.uidx()]


e_grid_GeV = prince_run_chain.cr_grid.grid
e_eV = e_grid_GeV * 1e9

chain_p = get_spectrum(solver_chain.state, solver_chain.spec_man, 2212)
expl_p = get_spectrum(solver_expl.state, solver_expl.spec_man, 2212)
expl_n = get_spectrum(solver_expl.state, solver_expl.spec_man, 2112)
if expl_n is None:
    expl_n = np.zeros_like(expl_p)
expl_total = expl_p + expl_n

chain_nuebar = get_spectrum(solver_chain.state, solver_chain.spec_man, -12)
expl_nuebar = get_spectrum(solver_expl.state, solver_expl.spec_man, -12)

print(f'energy grid: {e_eV.size} bins, {e_eV[0]:.2e} - {e_eV[-1]:.2e} eV')

## Plot spectra and ratios on the energy grid

Continuous lines across the full PriNCe energy grid (no down-sampling, no marker-only plotting). For the log-scale spectra panels we mask non-positive bins so the line breaks where the solver returned zero; the residual panels use a `where='valid'` mask only on bins where both runs are positive, but plot the line across every such bin.

In [ ]:
def masked(y, mask):
    """Return y with non-mask entries set to NaN — matplotlib draws a line
    break across NaN, so the visible curve covers exactly the masked region
    while remaining a single Line2D over the full grid."""
    out = np.full_like(y, np.nan, dtype=float)
    out[mask] = y[mask]
    return out


e2 = e_eV**2

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# --- Top-left: total nucleons ---
ax = axes[0, 0]
ax.loglog(e_eV, masked(e2 * chain_p, chain_p > 0), label='chain reducer (p)')
ax.loglog(e_eV, masked(e2 * expl_total, expl_total > 0), '--',
          label='explicit (p + n)')
if (expl_n > 0).any():
    ax.loglog(e_eV, masked(e2 * expl_n, expl_n > 0), ':', lw=1,
              label='explicit n (residual)')
ax.axvspan(1e16, 3e18, alpha=0.15, color='green', label='strict band')
ax.set_xlabel('E [eV]')
ax.set_ylabel('E² · spectrum [arb.]')
ax.set_title('Total nucleons at z=0')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

# --- Top-right: nucleon residual ---
ax = axes[0, 1]
nz = (chain_p > 0) & (expl_total > 0)
resid_n = np.full_like(chain_p, np.nan, dtype=float)
resid_n[nz] = (expl_total[nz] - chain_p[nz]) / chain_p[nz]
ax.semilogx(e_eV, resid_n)
ax.axhline(0, color='k', lw=0.5)
ax.axvspan(1e16, 3e18, alpha=0.15, color='green', label='strict band')
ax.axhspan(-0.01, 0.01, alpha=0.15, color='orange', label='±1% tol')
ax.set_xlabel('E [eV]')
ax.set_ylabel('(explicit - chain) / chain')
ax.set_title('Nucleon residual')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- Bottom-left: nu_e_bar spectrum ---
ax = axes[1, 0]
ax.loglog(e_eV, masked(e2 * chain_nuebar, chain_nuebar > 0),
          label='chain reducer')
ax.loglog(e_eV, masked(e2 * expl_nuebar, expl_nuebar > 0), '--',
          label='explicit')
ax.axvspan(1e15, 1e19, alpha=0.15, color='green', label='compare band')
ax.set_xlabel('E [eV]')
ax.set_ylabel('E² · spectrum [arb.]')
ax.set_title('ν̄_e at z=0')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

# --- Bottom-right: nu_e_bar residual ---
ax = axes[1, 1]
nz = (chain_nuebar > 0) & (expl_nuebar > 0)
resid_nu = np.full_like(chain_nuebar, np.nan, dtype=float)
resid_nu[nz] = (expl_nuebar[nz] - chain_nuebar[nz]) / chain_nuebar[nz]
ax.semilogx(e_eV, resid_nu)
ax.axhline(0, color='k', lw=0.5)
ax.axvspan(1e15, 1e19, alpha=0.15, color='green', label='compare band')
ax.axhspan(-0.01, 0.01, alpha=0.15, color='orange', label='±1% tol')
ax.set_xlabel('E [eV]')
ax.set_ylabel('(explicit - chain) / chain')
ax.set_title('ν̄_e residual')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

fig.suptitle(
    'Direct (chain-reducer) vs explicit decay — max_mass=4, He-4 source, z=1→0, scipy backend',
    fontsize=11,
)
fig.tight_layout()
plt.show()

## What to look for

- **Nucleon residual (top-right):** flat at <1 % across 1e16–3e18 eV, rising toward ~10–20 % above 3e18 eV as the chain-reducer's instant-decay approximation breaks down (γ_n ≳ 10⁹ → lab decay length approaches Mpc-Gpc, but chain-reducer still folds the decay in instantaneously).

- **ν̄_e residual (bottom-right):** ~factor-2 vertical offset is expected. Same physics, different sampling: the chain reducer evaluates the n→ν̄_e redistribution kernel on the cross-section's 200-bin x-grid (36 bins/dec) before folding into M; Λ_off acts on the 88-bin energy grid (8 bins/dec) directly. A bin-averaged construction in `_ensure_Lambda_split` would close the gap — tracked as a Step D follow-up.

- **Total flux band-integrated `expl/chain` ratio for ν̄_e** is typically ~0.5–0.6 in this configuration.